In [22]:
import numpy as np
import matplotlib.pyplot as plt

In [23]:

def function_1(x, y):
    r2 = x**2 + y**2
    term1 = np.power(r2, 0.25)
    term2 = np.sin(50 * np.power(r2, 0.1))**2 + 1
    return term1 * term2

def function_2(x, y):
    return (1 - x)**2 + 100 * (y - x**2)**2

In [ ]:
class PSO():
    def __init__(self, c1=2, c2=2, w=0.7, strategy='global'):
        self.c1 = c1
        self.c2 = c2
        self.w = w
        self.strategy = strategy
        
    def run(self, function, n_particles, bounds, max_iters=1000):
        """
        bounds = [(x_min, x_max), (y_min, y_max), ...]
        """
        self.bounds = np.array(bounds)
        dim = len(bounds)
        
        #inicializamos las posiciones
        positions = np.random.uniform(
            low=self.bounds[:, 0],  
            high=self.bounds[:, 1],
            size=(n_particles, dim)
        )
        #incializamos las velocidades nulas
        velocities = np.zeros((n_particles, dim))
        
        #calculasmo los fitnes para esas posiciones
        fitness = np.array([function(*p) for p in positions])
        
        pbest_pos = positions.copy() #personal best positions
        pbest_fit = fitness.copy() #personal best fintess
        
        gbest_idx = np.argmin(pbest_fit) #global best index
        gbest_pos = pbest_pos[gbest_idx].copy() #global best position
        gbest_fit = pbest_fit[gbest_idx] #global best fit
        
        #bucle principal
        for _ in range(max_iters):
            #actualizamos segun la estrategia
            for i in range(n_particles):
                if self.strategy == 'local':
                    self._update_local(i, positions, velocities, pbest_pos, pbest_fit)
                elif self.strategy == 'global':
                    self._update_global(i, positions, velocities, pbest_pos, gbest_pos)
            
            fitness = np.array([function(*p) for p in positions]) #recalculamos los fitness
            
            # actulizamos mejores personales y globales
            for i in range(n_particles):
                if fitness[i] < pbest_fit[i]:
                    pbest_fit[i] = fitness[i]
                    pbest_pos[i] = positions[i].copy()
                    
                    if fitness[i] < gbest_fit:
                        gbest_fit = fitness[i]
                        gbest_pos = positions[i].copy()  # ✓ corregido typo
        
        # retornamos
        return gbest_pos, gbest_fit
    
    def _update_global(self, i, positions, velocities, pbest_pos, gbest_pos):
        dim = len(positions[i])
        r1 = np.random.random(dim)
        r2 = np.random.random(dim)
        
        #actualizamos velocidad y posicion
        velocities[i] = (self.w * velocities[i] + 
                         self.c1 * r1 * (pbest_pos[i] - positions[i]) + 
                         self.c2 * r2 * (gbest_pos - positions[i]))
        
        positions[i] += velocities[i]
        positions[i] = np.clip(positions[i], self.bounds[:, 0], self.bounds[:, 1])
    
    def _update_local(self, i, positions, velocities, pbest_pos, pbest_fit):
        """
        cadaparticula mira el lider de su vecindad
        """
        dim = len(positions[i])
        n_particles = len(positions)
        
        #toplogia de vecindad de radio 1
        left = (i - 1) % n_particles
        right = (i + 1) % n_particles
        
        # mejor de la vecindad
        neighborhood = [left, i, right] #definimos la vecindad

        #buscamos el indice con menor pbest_fit

        best_idx = neighborhood[0] 
        best_fitness = pbest_fit[best_idx]
        for idx in neighborhood:
            if pbest_fit[idx]<best_fitness:
                best_idx = idx
                best_fitness = pbest_fit[idx]
                
        #ya tenemos el lider de la vecindad
        lbest_pos = pbest_pos[best_idx].copy()
        
        r1 = np.random.random(dim)
        r2 = np.random.random(dim)
        
        velocities[i] = (self.w * velocities[i] + 
                         self.c1 * r1 * (pbest_pos[i] - positions[i]) + 
                         self.c2 * r2 * (lbest_pos - positions[i]))
        
        positions[i] += velocities[i]
        positions[i] = np.clip(positions[i], self.bounds[:, 0], self.bounds[:, 1])




In [25]:
pso = PSO(c1=2, c2=2, strategy='global')
best_pos, best_fit = pso.run(function_1, n_particles=30, bounds=[(-10, 10), (-10, 10)], max_iters=1000)
print(f"Mejor posición: {best_pos}, Mejor fitness: {best_fit}")

Mejor posición: [-1.06606904e-27  2.05200736e-28], Mejor fitness: 3.2949027258520926e-14
